# NB12：GCP 上的 RAG 系統 — Vertex AI、Vector Search 與 Cloud Run 部署

## 學習目標

| 主題 | 說明 |
|------|------|
| Gemini API | 透過 Vertex AI 使用 Gemini，與 OpenAI 比較 |
| Vertex AI Vector Search | 企業級 HNSW 向量搜索服務 |
| Vertex AI RAG Engine | Google 託管的 RAG 管線 |
| Cloud Run 部署 | 將 RAG 系統包裝為 API 並部署 |
| Vertex AI Pipelines | 自動化重新索引流程 |
| Google ADK | Agent Development Kit 基礎 |

## 📋 使用說明

本筆記本有兩種執行模式：

- **🌐 GCP 模式**：需要 GCP 專案與認證，標記為 `[GCP]`
- **💻 本地模式**：使用 OpenAI + ChromaDB，無需 GCP，標記為 `[本地]`

預設使用本地模式（`USE_GCP=false`），大多數學員可直接執行。

---
**系列位置**：NB01 → NB02 → ... → NB09 → **NB12（本筆記本）**

**前置知識**：建議先完成 NB01（RAG 基礎）與 NB07（生產優化）

---
## Part 0：環境設定

### 兩種模式的套件需求

```
本地模式（預設）：openai, chromadb, fastapi, uvicorn
GCP 模式：google-cloud-aiplatform, vertexai, google-auth
```

In [ ]:
# 安裝本地模式所需套件
# !pip install openai chromadb fastapi uvicorn python-dotenv

# 若要使用 GCP 模式，另外安裝：
# !pip install google-cloud-aiplatform vertexai google-auth

print("套件確認完畢")

In [ ]:
import os

# ======================================================
# 模式設定：將 USE_GCP 改為 True 以啟用 GCP 功能
# ======================================================
USE_GCP = os.getenv("USE_GCP", "false").lower() == "true"

# GCP 設定（USE_GCP=True 時才需要）
GCP_PROJECT_ID = os.getenv("GCP_PROJECT_ID", "your-project-id")
GCP_LOCATION   = os.getenv("GCP_LOCATION", "us-central1")

# OpenAI 設定（本地模式使用）
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-openai-key")

print(f"執行模式：{'🌐 GCP 模式' if USE_GCP else '💻 本地模式'}")
if USE_GCP:
    print(f"  GCP 專案：{GCP_PROJECT_ID}")
    print(f"  區域：{GCP_LOCATION}")

### 🌐 GCP 認證方式（需要 GCP 專案與認證）

若要使用 GCP 模式，請先完成以下認證步驟：

**方法 1：本地開發（推薦）**
```bash
# 安裝 gcloud CLI
# https://cloud.google.com/sdk/docs/install

# 登入並設定 Application Default Credentials
gcloud auth application-default login

# 設定專案
gcloud config set project YOUR_PROJECT_ID
```

**方法 2：Service Account（生產環境）**
```bash
# 下載 Service Account JSON 金鑰
export GOOGLE_APPLICATION_CREDENTIALS="/path/to/service-account.json"
```

**方法 3：在 Vertex AI Workbench 或 Colab Enterprise 中執行**
```python
# 認證會自動處理，不需額外設定
```

### 所需 IAM 權限
```
roles/aiplatform.user          # Vertex AI 使用者
roles/storage.objectAdmin      # GCS 存取（用於 RAG Engine）
roles/run.developer            # Cloud Run 部署
```

In [ ]:
if USE_GCP:
    try:
        import google.auth
        credentials, project = google.auth.default()
        print(f"✅ GCP 認證成功")
        print(f"   專案：{project}")
    except Exception as e:
        print(f"❌ GCP 認證失敗：{e}")
        print("   請執行：gcloud auth application-default login")
else:
    print("💻 本地模式：跳過 GCP 認證")

---
## Part 1：Gemini API vs OpenAI API

### 主要差異比較

| 特性 | Gemini 1.5 Flash (Vertex AI) | GPT-4o-mini (OpenAI) |
|------|------------------------------|----------------------|
| Context Window | 1,000,000 tokens | 128,000 tokens |
| 多模態輸入 | 文字、圖片、影片、音訊、PDF | 文字、圖片 |
| 定價模式 | 按 token 計費（Google Cloud） | 按 token 計費（OpenAI） |
| 認證方式 | Google ADC / Service Account | API Key |
| Tool Calling | FunctionDeclaration 語法 | JSON Schema 語法 |
| 企業功能 | VPC、CMEK、資料不離 GCP | Enterprise 方案 |
| 延遲 | Flash 版約 200-500ms | Mini 版約 300-600ms |

### 架構差異

```
[OpenAI 路徑]
應用程式 → OpenAI API → GPT 模型 → 回應

[Vertex AI 路徑]
應用程式 → Vertex AI Endpoint（Google Cloud）→ Gemini 模型 → 回應
              ↑
         資料留在您的 GCP 專案中（更符合企業合規需求）
```

In [ ]:
# 準備測試資料（兩種模式共用）
SAMPLE_CONTEXT = """
台灣的半導體產業是全球供應鏈的關鍵環節。
台積電（TSMC）是全球最大的晶圓代工廠，負責製造 Apple、NVIDIA、AMD 等公司的晶片。
台灣在先進製程節點（如 3nm、5nm）擁有領先優勢。
2023 年，台灣半導體出口額超過 1,500 億美元，佔全國出口的 40% 以上。
"""

SAMPLE_QUESTION = "台積電在全球半導體產業中扮演什麼角色？"

RAG_PROMPT_TEMPLATE = """根據以下背景資訊回答問題。

背景資訊：
{context}

問題：{question}

請用繁體中文回答，並只根據提供的資訊作答。"""

In [ ]:
# ============================================================
# [GCP] 使用 Vertex AI SDK 呼叫 Gemini
# 需要 GCP 專案與認證
# ============================================================

if USE_GCP:
    import vertexai
    from vertexai.generative_models import GenerativeModel, Part

    # 初始化 Vertex AI
    vertexai.init(project=GCP_PROJECT_ID, location=GCP_LOCATION)

    # 建立 Gemini 模型實例
    gemini_model = GenerativeModel("gemini-1.5-flash")

    # 組合 RAG prompt
    prompt = RAG_PROMPT_TEMPLATE.format(
        context=SAMPLE_CONTEXT,
        question=SAMPLE_QUESTION
    )

    # 呼叫 Gemini
    response = gemini_model.generate_content(prompt)
    print("=== Gemini 1.5 Flash 回應 ===")
    print(response.text)
    print(f"\n使用 tokens：{response.usage_metadata}")

In [ ]:
# ============================================================
# [GCP] Gemini 多模態輸入範例（文字 + 圖片）
# 需要 GCP 專案與認證
# ============================================================

if USE_GCP:
    from vertexai.generative_models import GenerativeModel, Part, Image
    import base64

    gemini_vision = GenerativeModel("gemini-1.5-flash")

    # 從 GCS 載入圖片（實際使用時替換路徑）
    # image_part = Part.from_uri("gs://your-bucket/image.png", mime_type="image/png")

    # 或從本地檔案載入
    # with open("diagram.png", "rb") as f:
    #     image_data = f.read()
    # image_part = Part.from_data(image_data, mime_type="image/png")

    # 多模態請求範例（此處只示範結構，不實際執行）
    multimodal_request_example = [
        "請分析這張架構圖並說明 RAG 系統的資料流：",
        # image_part,  # 取消此行註解以實際傳入圖片
        "圖中有哪些主要元件？"
    ]

    print("多模態請求結構（示範）：")
    print("  - 文字部分：說明與問題")
    print("  - 圖片部分：架構圖、截圖、PDF 頁面等")
    print("\nGemini 1M token 視窗讓長文件處理成為可能")
    print("例如：直接傳入整本技術手冊（PDF）進行 RAG")

In [ ]:
# ============================================================
# [本地] 使用 OpenAI 作為本地模擬（功能等價）
# ============================================================

if not USE_GCP:
    from openai import OpenAI

    client = OpenAI(api_key=OPENAI_API_KEY)

    prompt = RAG_PROMPT_TEMPLATE.format(
        context=SAMPLE_CONTEXT,
        question=SAMPLE_QUESTION
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )

    answer = response.choices[0].message.content
    print("=== GPT-4o-mini 回應（本地模擬）===")
    print(answer)
    print(f"\n使用 tokens：{response.usage.total_tokens}")

### Tool Calling 語法比較

**OpenAI 語法：**
```python
tools = [{
    "type": "function",
    "function": {
        "name": "search_docs",
        "description": "搜尋文件庫",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"}
            }
        }
    }
}]
```

**Gemini (Vertex AI) 語法：**
```python
from vertexai.generative_models import FunctionDeclaration, Tool

search_docs_func = FunctionDeclaration(
    name="search_docs",
    description="搜尋文件庫",
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "搜尋關鍵字"}
        }
    }
)

rag_tool = Tool(function_declarations=[search_docs_func])
model = GenerativeModel("gemini-1.5-flash", tools=[rag_tool])
```

---
## Part 2：Vertex AI Vector Search

### 什麼是 Vertex AI Vector Search？

Vertex AI Vector Search（前身 Matching Engine）是 Google 的**企業級託管向量搜索服務**：

```
特點：
✅ 託管 HNSW 演算法（無需自行維護索引）
✅ 自動水平擴展（處理數十億向量）
✅ 低延遲查詢（< 10ms at p99）
✅ 多租戶支援（一個 Index 服務多個應用）
✅ 與其他 GCP 服務整合（Cloud Storage、Pub/Sub）

對比 ChromaDB（本地/自託管）：
❌ 費用較高（按節點時間計費）
❌ 需要 GCP 專案
✅ 適合 10M+ 向量的企業場景
```

### 企業 RAG 架構圖

```
┌─────────────────────────────────────────────────────┐
│                  GCP 企業 RAG 架構                   │
├─────────────────────────────────────────────────────┤
│                                                     │
│  ┌─────────┐    ┌──────────────┐    ┌───────────┐  │
│  │ 文件來源 │───▶│ Vertex AI    │───▶│  Cloud    │  │
│  │ (GCS)   │    │  Pipelines   │    │  Storage  │  │
│  └─────────┘    │ (embedding)  │    │ (vectors) │  │
│                 └──────────────┘    └─────┬─────┘  │
│                                           │upsert  │
│  ┌─────────┐    ┌──────────────┐    ┌─────▼─────┐  │
│  │ 使用者  │───▶│  Cloud Run   │───▶│ Vertex AI │  │
│  │ 查詢    │    │  (RAG API)   │    │  Vector   │  │
│  └─────────┘    └──────┬───────┘    │  Search   │  │
│                        │            └───────────┘  │
│                        ▼                           │
│                 ┌──────────────┐                   │
│                 │  Gemini API  │                   │
│                 │ (生成回答)   │                   │
│                 └──────────────┘                   │
└─────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# [GCP] Vertex AI Vector Search — 完整流程
# 需要 GCP 專案與認證
# 注意：建立 Index 需要 30-60 分鐘，實際部署建議在 CI/CD 中執行
# ============================================================

if USE_GCP:
    from google.cloud import aiplatform

    aiplatform.init(project=GCP_PROJECT_ID, location=GCP_LOCATION)

    # ── 步驟 1：建立向量索引 ──────────────────────────────────
    print("步驟 1：建立 Vector Search Index（約需 30-60 分鐘）")

    # 索引設定
    index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
        display_name="rag-knowledge-base-index",
        contents_delta_uri="gs://your-bucket/embeddings/",  # 向量資料路徑
        dimensions=1536,          # text-embedding-ada-002 維度
        approximate_neighbors_count=150,
        distance_measure_type="DOT_PRODUCT_DISTANCE",
        leaf_node_embedding_count=500,
        leaf_nodes_to_search_percent=7,
        description="RAG 知識庫向量索引",
    )
    print(f"   Index 建立中：{index.resource_name}")

    # ── 步驟 2：建立 Index Endpoint ────────────────────────────
    print("\n步驟 2：建立 Index Endpoint")

    index_endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
        display_name="rag-index-endpoint",
        public_endpoint_enabled=True,  # 公開端點（生產環境建議用 VPC）
    )
    print(f"   Endpoint：{index_endpoint.resource_name}")

    # ── 步驟 3：部署 Index 到 Endpoint ────────────────────────
    print("\n步驟 3：部署 Index（約需 10-20 分鐘）")

    index_endpoint.deploy_index(
        index=index,
        deployed_index_id="rag_knowledge_base",
        display_name="RAG 知識庫",
        machine_type="e2-standard-16",
        min_replica_count=1,
        max_replica_count=5,  # 自動擴展
    )
    print("   Index 部署完成")

In [ ]:
# ============================================================
# [GCP] Vertex AI Vector Search — 查詢
# 需要 GCP 專案與認證
# ============================================================

if USE_GCP:
    import numpy as np

    # 假設已有部署好的 endpoint（替換為實際 ID）
    ENDPOINT_ID = "your-endpoint-id"
    DEPLOYED_INDEX_ID = "rag_knowledge_base"

    # 取得 endpoint
    endpoint = aiplatform.MatchingEngineIndexEndpoint(
        index_endpoint_name=f"projects/{GCP_PROJECT_ID}/locations/{GCP_LOCATION}/indexEndpoints/{ENDPOINT_ID}"
    )

    # 查詢向量（實際使用時應由 embedding 模型生成）
    query_embedding = np.random.rand(1536).tolist()  # 示範用隨機向量

    # 執行向量搜索
    results = endpoint.find_neighbors(
        deployed_index_id=DEPLOYED_INDEX_ID,
        queries=[query_embedding],
        num_neighbors=5,             # 取最近 5 個結果
        filter=[                     # 支援元資料過濾（如過濾特定文件類型）
            aiplatform.matching_engine.matching_engine_index_endpoint.Namespace(
                name="doc_type",
                allow_tokens=["technical", "manual"]
            )
        ]
    )

    print("Vector Search 查詢結果：")
    for neighbor in results[0]:
        print(f"  ID: {neighbor.id}, 距離: {neighbor.distance:.4f}")

In [ ]:
# ============================================================
# [本地] ChromaDB 模擬 Vector Search
# ============================================================

if not USE_GCP:
    import chromadb
    from openai import OpenAI

    client = OpenAI(api_key=OPENAI_API_KEY)
    chroma_client = chromadb.Client()

    # 建立集合（對應 Vertex AI 的 Index）
    collection = chroma_client.get_or_create_collection(
        name="rag_knowledge_base",
        metadata={"hnsw:space": "cosine"}  # 使用 cosine 相似度
    )

    # 準備文件
    documents = [
        "台積電是全球最大的晶圓代工廠，負責製造 Apple、NVIDIA 等公司的晶片。",
        "台灣半導體出口額超過 1,500 億美元，佔全國出口的 40% 以上。",
        "台積電在先進製程節點（3nm、5nm）擁有全球領先優勢。",
        "南韓三星是台積電在先進製程的主要競爭對手。",
        "美國英特爾也正在積極發展晶圓代工業務。"
    ]

    # 生成 embedding 並存入
    def get_embedding(text):
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding

    print("正在生成 embeddings 並索引...")
    for i, doc in enumerate(documents):
        embedding = get_embedding(doc)
        collection.upsert(
            ids=[f"doc_{i}"],
            embeddings=[embedding],
            documents=[doc],
            metadatas=[{"doc_type": "technical", "index": i}]
        )

    # 查詢
    query = "台積電的市場地位"
    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        where={"doc_type": "technical"}  # 元資料過濾
    )

    print(f"\n查詢：{query}")
    print("\n最相關文件：")
    for doc, distance in zip(results["documents"][0], results["distances"][0]):
        print(f"  距離 {distance:.4f}：{doc[:50]}...")

---
## Part 3：Vertex AI RAG Engine

### 什麼是 Vertex AI RAG Engine？

Google 在 2024 年推出的**完全託管 RAG 管線服務**，整合了：
- 文件解析（PDF、Word、HTML）
- 自動切塊（Auto Chunking）
- Embedding 生成
- 向量索引管理
- 檢索 + 生成一體化

### DIY RAG vs Managed RAG Engine 比較

| 面向 | DIY RAG（NB01-07）| Vertex AI RAG Engine |
|------|-------------------|----------------------|
| 控制彈性 | 完全自訂 | 有限設定 |
| 維運成本 | 高（需自行維護）| 低（全託管）|
| 開發速度 | 慢 | 快 |
| 可客製化 | 高（切塊、reranker）| 中等 |
| 費用透明度 | 高 | 依使用量計費 |
| 適合場景 | 需要特殊優化的系統 | 快速 PoC 或標準 RAG |
| 除錯難度 | 低（全程可見）| 高（黑箱部分較多）|

In [ ]:
# ============================================================
# [GCP] Vertex AI RAG Engine — 建立語料庫與索引
# 需要 GCP 專案與認證
# ============================================================

if USE_GCP:
    import vertexai
    from vertexai.preview import rag
    from vertexai.preview.generative_models import GenerativeModel, Tool

    vertexai.init(project=GCP_PROJECT_ID, location=GCP_LOCATION)

    # ── 步驟 1：建立 RAG Corpus（知識庫）─────────────────────
    print("步驟 1：建立 RAG Corpus")

    corpus = rag.create_corpus(
        display_name="taiwan-semiconductor-corpus",
        description="台灣半導體產業知識庫"
    )
    print(f"   Corpus 建立：{corpus.name}")

    # ── 步驟 2：從 GCS 匯入文件 ────────────────────────────────
    print("\n步驟 2：從 GCS 匯入文件")

    # 支援格式：PDF、Google Docs、TXT、HTML
    response = rag.import_files(
        corpus_name=corpus.name,
        paths=[
            "gs://your-bucket/documents/semiconductor_report.pdf",
            "gs://your-bucket/documents/",  # 支援整個資料夾
        ],
        transformation_config=rag.TransformationConfig(
            chunking_config=rag.ChunkingConfig(
                chunk_size=512,
                chunk_overlap=100,
            )
        ),
    )
    print(f"   匯入結果：{response}")

    # ── 步驟 3：查詢 RAG Corpus ────────────────────────────────
    print("\n步驟 3：查詢 RAG Corpus")

    rag_response = rag.retrieval_query(
        rag_resources=[
            rag.RagResource(
                rag_corpus=corpus.name,
            )
        ],
        text="台積電的先進製程優勢",
        similarity_top_k=5,
        vector_distance_threshold=0.5,
    )

    print("\n檢索結果：")
    for context in rag_response.contexts.contexts:
        print(f"  來源：{context.source_uri}")
        print(f"  內容：{context.text[:100]}...")
        print(f"  相關度：{context.score:.4f}")

In [ ]:
# ============================================================
# [GCP] Vertex AI RAG Engine — 整合生成（RAG + Gemini）
# 需要 GCP 專案與認證
# ============================================================

if USE_GCP:
    from vertexai.preview.generative_models import GenerativeModel, Tool
    from vertexai.preview import rag

    # 建立 RAG 工具
    rag_retrieval_tool = Tool.from_retrieval(
        retrieval=rag.Retrieval(
            source=rag.VertexRagStore(
                rag_resources=[
                    rag.RagResource(rag_corpus=corpus.name)
                ],
                similarity_top_k=5,
            )
        )
    )

    # Gemini + RAG 整合模型
    rag_model = GenerativeModel(
        model_name="gemini-1.5-flash",
        tools=[rag_retrieval_tool]
    )

    # 一行完成 RAG 查詢 + 生成
    response = rag_model.generate_content("台積電在 3nm 製程上有什麼優勢？")
    print("=== RAG Engine 生成結果 ===")
    print(response.text)

In [ ]:
# ============================================================
# [本地] 模擬 Vertex AI RAG Engine 完整流程
# 使用 NB07 風格的自訂 RAG 管線
# ============================================================

if not USE_GCP:
    import chromadb
    from openai import OpenAI

    client = OpenAI(api_key=OPENAI_API_KEY)
    chroma = chromadb.Client()

    class LocalRAGEngine:
        """模擬 Vertex AI RAG Engine 的本地版本"""

        def __init__(self, corpus_name: str):
            self.corpus_name = corpus_name
            self.collection = chroma.get_or_create_collection(corpus_name)
            self.client = OpenAI(api_key=OPENAI_API_KEY)

        def import_files(self, texts: list[str], chunk_size: int = 512):
            """模擬 rag.import_files()"""
            for i, text in enumerate(texts):
                # 簡單切塊
                chunks = [text[j:j+chunk_size] for j in range(0, len(text), chunk_size-50)]
                for k, chunk in enumerate(chunks):
                    emb = self.client.embeddings.create(
                        model="text-embedding-3-small", input=chunk
                    ).data[0].embedding
                    self.collection.upsert(
                        ids=[f"doc_{i}_chunk_{k}"],
                        embeddings=[emb],
                        documents=[chunk]
                    )
            print(f"   已匯入 {len(texts)} 份文件")

        def retrieval_query(self, text: str, top_k: int = 3):
            """模擬 rag.retrieval_query()"""
            emb = self.client.embeddings.create(
                model="text-embedding-3-small", input=text
            ).data[0].embedding
            results = self.collection.query(query_embeddings=[emb], n_results=top_k)
            return results["documents"][0]

        def generate_content(self, question: str):
            """模擬 RAG Engine 的 generate_content()"""
            contexts = self.retrieval_query(question)
            context_text = "\n".join(contexts)
            prompt = RAG_PROMPT_TEMPLATE.format(
                context=context_text, question=question
            )
            response = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}]
            )
            return response.choices[0].message.content

    # 使用本地 RAG Engine
    engine = LocalRAGEngine("taiwan-semiconductor-corpus")

    docs = [
        "台積電（TSMC）是全球最大的晶圓代工廠。2023年，台積電在先進製程（3nm/5nm）的市佔率超過 90%。",
        "台積電的 3nm 製程比 Samsung 同期製程效能高 10-15%，功耗低 25-30%。",
        "台積電在美國亞利桑那州、日本熊本縣、德國德勒斯登建設新廠，實現地緣政治多元化布局。"
    ]

    print("匯入文件...")
    engine.import_files(docs)

    print("\n執行 RAG 查詢...")
    answer = engine.generate_content("台積電在 3nm 製程上有什麼優勢？")
    print("\n=== 本地 RAG Engine 結果 ===")
    print(answer)

---
## Part 4：Cloud Run 部署 RAG API

### 架構說明

```
┌─────────────────────────────────────────────────┐
│              Cloud Run RAG API                  │
│                                                 │
│  POST /query                                    │
│  ┌─────────────────────────────────────────┐    │
│  │ 1. 接收查詢請求                          │    │
│  │ 2. 生成查詢 embedding                   │    │
│  │ 3. 向量搜索（Vector Search）             │    │
│  │ 4. 組合 prompt + 呼叫 Gemini            │    │
│  │ 5. 回傳結構化回應                        │    │
│  └─────────────────────────────────────────┘    │
│                                                 │
│  GET /health → {"status": "ok"}                │
└─────────────────────────────────────────────────┘
        ↑
  Cloud Run 特性：
  • 自動 Scale to Zero（不用時不收費）
  • 最大並發請求：1000
  • 最大記憶體：32GB
  • 請求逾時：60 分鐘
```

In [ ]:
# ============================================================
# FastAPI RAG 應用程式（本地執行 + Cloud Run 部署皆使用此程式碼）
# 儲存為 main.py 後可直接部署
# ============================================================

FASTAPI_APP_CODE = '''
# main.py — RAG API for Cloud Run
import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional

app = FastAPI(
    title="RAG API",
    description="GCP Vertex AI RAG 查詢服務",
    version="1.0.0"
)

USE_GCP = os.getenv("USE_GCP", "false").lower() == "true"
GCP_PROJECT_ID = os.getenv("GCP_PROJECT_ID", "")
GCP_LOCATION = os.getenv("GCP_LOCATION", "us-central1")

# ── 請求/回應模型 ──────────────────────────────────────────────
class QueryRequest(BaseModel):
    question: str
    top_k: int = 3
    filter_metadata: Optional[dict] = None

class SourceDocument(BaseModel):
    content: str
    score: float
    source: Optional[str] = None

class QueryResponse(BaseModel):
    answer: str
    sources: list[SourceDocument]
    model: str
    latency_ms: float

# ── 端點 ─────────────────────────────────────────────────────
@app.get("/health")
async def health_check():
    return {"status": "ok", "mode": "gcp" if USE_GCP else "local"}

@app.post("/query", response_model=QueryResponse)
async def query_rag(request: QueryRequest):
    import time
    start = time.time()

    try:
        if USE_GCP:
            answer, sources = await _query_vertex_rag(request)
            model_name = "gemini-1.5-flash"
        else:
            answer, sources = await _query_local_rag(request)
            model_name = "gpt-4o-mini"

        latency = (time.time() - start) * 1000

        return QueryResponse(
            answer=answer,
            sources=sources,
            model=model_name,
            latency_ms=round(latency, 2)
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

async def _query_vertex_rag(request: QueryRequest):
    """GCP 版本：使用 Vertex AI RAG Engine"""
    import vertexai
    from vertexai.preview import rag
    from vertexai.preview.generative_models import GenerativeModel, Tool

    vertexai.init(project=GCP_PROJECT_ID, location=GCP_LOCATION)
    CORPUS_NAME = os.getenv("RAG_CORPUS_NAME", "")

    rag_tool = Tool.from_retrieval(
        retrieval=rag.Retrieval(
            source=rag.VertexRagStore(
                rag_resources=[rag.RagResource(rag_corpus=CORPUS_NAME)],
                similarity_top_k=request.top_k,
            )
        )
    )
    model = GenerativeModel("gemini-1.5-flash", tools=[rag_tool])
    response = model.generate_content(request.question)

    sources = [SourceDocument(content="來自 Vertex AI RAG Engine", score=1.0)]
    return response.text, sources

async def _query_local_rag(request: QueryRequest):
    """本地版本：使用 OpenAI + ChromaDB"""
    from openai import OpenAI
    import chromadb

    oai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    chroma = chromadb.Client()
    col = chroma.get_or_create_collection("rag_knowledge_base")

    emb = oai.embeddings.create(
        model="text-embedding-3-small", input=request.question
    ).data[0].embedding

    results = col.query(query_embeddings=[emb], n_results=request.top_k)
    docs = results["documents"][0]
    distances = results["distances"][0]

    context = "\\n".join(docs)
    prompt = f"根據以下資訊回答：\\n{context}\\n\\n問題：{request.question}"

    resp = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    sources = [
        SourceDocument(content=doc, score=1 - dist)
        for doc, dist in zip(docs, distances)
    ]
    return resp.choices[0].message.content, sources

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=int(os.getenv("PORT", 8080)))
'''

# 將程式碼寫入檔案（可選）
# with open("main.py", "w") as f:
#     f.write(FASTAPI_APP_CODE)

print("FastAPI 應用程式程式碼已準備好")
print("端點：")
print("  GET  /health  → 健康檢查")
print("  POST /query   → RAG 查詢")

In [ ]:
# ============================================================
# Dockerfile（適用於 Cloud Run）
# ============================================================

DOCKERFILE = '''
# Dockerfile
FROM python:3.11-slim

WORKDIR /app

# 複製需求檔
COPY requirements.txt .

# 安裝套件
RUN pip install --no-cache-dir -r requirements.txt

# 複製應用程式
COPY main.py .

# Cloud Run 使用 PORT 環境變數
ENV PORT=8080
EXPOSE 8080

# 啟動應用程式
CMD ["python", "main.py"]
'''

REQUIREMENTS = '''
# requirements.txt
fastapi==0.111.0
uvicorn[standard]==0.30.0
pydantic==2.7.0
openai==1.30.0
chromadb==0.5.0
# GCP 模式需要：
# google-cloud-aiplatform==1.50.0
# vertexai==1.50.0
'''

print("Dockerfile 內容：")
print(DOCKERFILE)
print("\nrequirements.txt 內容：")
print(REQUIREMENTS)

In [ ]:
# ============================================================
# [GCP] Cloud Run 部署指令
# 需要 GCP 專案與認證
# ============================================================

DEPLOY_COMMANDS = f"""
# ── 步驟 1：設定環境變數 ─────────────────────────────────────
export PROJECT_ID="{GCP_PROJECT_ID}"
export REGION="{GCP_LOCATION}"
export SERVICE_NAME="rag-api"
export IMAGE="gcr.io/$PROJECT_ID/$SERVICE_NAME"

# ── 步驟 2：建構 Docker 映像檔 ───────────────────────────────
docker build -t $IMAGE .

# ── 步驟 3：推送到 Artifact Registry ────────────────────────
gcloud auth configure-docker
docker push $IMAGE

# ── 步驟 4：部署到 Cloud Run ─────────────────────────────────
gcloud run deploy $SERVICE_NAME \\
  --image $IMAGE \\
  --platform managed \\
  --region $REGION \\
  --memory 2Gi \\
  --cpu 2 \\
  --concurrency 100 \\
  --max-instances 10 \\
  --timeout 60 \\
  --set-env-vars USE_GCP=true,GCP_PROJECT_ID=$PROJECT_ID \\
  --allow-unauthenticated  # 生產環境應移除此選項，改用 IAM

# ── 步驟 5：取得服務 URL ─────────────────────────────────────
SERVICE_URL=$(gcloud run services describe $SERVICE_NAME \\
  --region $REGION \\
  --format 'value(status.url)')
echo "服務 URL：$SERVICE_URL"

# ── 步驟 6：測試 API ─────────────────────────────────────────
curl -X POST "$SERVICE_URL/query" \\
  -H "Content-Type: application/json" \\
  -d '{{"question": "台積電的市場地位？", "top_k": 3}}'
"""

print("Cloud Run 部署指令：")
print(DEPLOY_COMMANDS)

In [ ]:
# ============================================================
# [本地] 在 Notebook 中測試 FastAPI
# ============================================================

if not USE_GCP:
    # 示範 FastAPI 請求/回應結構（不實際啟動服務器）
    import json

    # 模擬請求
    sample_request = {
        "question": "台積電在 3nm 製程上的競爭優勢？",
        "top_k": 3,
        "filter_metadata": None
    }

    # 模擬回應
    sample_response = {
        "answer": "台積電在 3nm 製程上擁有...",
        "sources": [
            {"content": "台積電 3nm 比三星效能高 10-15%", "score": 0.92, "source": "doc_0"},
            {"content": "台積電全球晶圓代工市佔率超過 60%", "score": 0.87, "source": "doc_1"}
        ],
        "model": "gpt-4o-mini",
        "latency_ms": 342.5
    }

    print("API 請求範例：")
    print(json.dumps(sample_request, ensure_ascii=False, indent=2))
    print("\nAPI 回應範例：")
    print(json.dumps(sample_response, ensure_ascii=False, indent=2))

    print("\n本地啟動指令：")
    print("  uvicorn main:app --reload --port 8080")
    print("  http://localhost:8080/docs  # Swagger UI")

---
## Part 5：Vertex AI Pipelines — 自動化重新索引

### 為什麼需要自動化重新索引？

```
問題場景：
  Day 1：知識庫有 10,000 份文件 → 建立向量索引
  Day 30：新增 500 份新文件，修改 200 份舊文件
  
  如果不重新索引：RAG 系統回答過時資訊！

解決方案：定期自動重新索引
  ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
  │ 偵測文件變更 │────▶│ 重新 embed  │────▶│ 更新向量索引 │
  └─────────────┘     └─────────────┘     └─────────────┘
          ↑
  Cloud Scheduler 每日觸發
```

### Vertex AI Pipelines 架構

```
Cloud Scheduler (每日 02:00)
       │
       ▼
Vertex AI Pipelines
       │
       ├── Component 1: load_documents
       │     從 GCS 讀取新/修改的文件
       │
       ├── Component 2: chunk_documents
       │     切塊（512 tokens, 50 token overlap）
       │
       ├── Component 3: generate_embeddings
       │     批次生成向量（並行處理）
       │
       └── Component 4: upsert_to_vector_search
             更新 Vertex AI Vector Search 索引
```

In [ ]:
# ============================================================
# [GCP] Vertex AI Pipelines — 重新索引管線
# 需要 GCP 專案與認證
# ============================================================

if USE_GCP:
    from kfp import dsl
    from kfp.dsl import component, pipeline, Output, Artifact
    from google.cloud import aiplatform

    # ── Component 1：載入文件 ──────────────────────────────────
    @component(
        base_image="python:3.11-slim",
        packages_to_install=["google-cloud-storage"]
    )
    def load_documents(
        gcs_bucket: str,
        gcs_prefix: str,
        documents: Output[Artifact]
    ):
        from google.cloud import storage
        import json

        client = storage.Client()
        bucket = client.bucket(gcs_bucket)
        blobs = bucket.list_blobs(prefix=gcs_prefix)

        docs = []
        for blob in blobs:
            if blob.name.endswith(".txt"):
                content = blob.download_as_text()
                docs.append({"id": blob.name, "content": content})

        with open(documents.path, "w") as f:
            json.dump(docs, f)

        print(f"載入 {len(docs)} 份文件")

    # ── Component 2：切塊 ─────────────────────────────────────
    @component(base_image="python:3.11-slim")
    def chunk_documents(
        documents: dsl.Input[Artifact],
        chunks: Output[Artifact],
        chunk_size: int = 512,
        overlap: int = 50
    ):
        import json

        with open(documents.path) as f:
            docs = json.load(f)

        all_chunks = []
        for doc in docs:
            text = doc["content"]
            for i in range(0, len(text), chunk_size - overlap):
                chunk_text = text[i:i + chunk_size]
                if len(chunk_text) > 50:
                    all_chunks.append({
                        "id": f"{doc['id']}_chunk_{i}",
                        "text": chunk_text,
                        "source": doc["id"]
                    })

        with open(chunks.path, "w") as f:
            json.dump(all_chunks, f)

        print(f"產生 {len(all_chunks)} 個切塊")

    # ── Component 3：生成 Embeddings ──────────────────────────
    @component(
        base_image="python:3.11-slim",
        packages_to_install=["google-cloud-aiplatform"]
    )
    def generate_embeddings(
        chunks: dsl.Input[Artifact],
        embeddings: Output[Artifact],
        project_id: str,
        location: str
    ):
        import json
        from google.cloud import aiplatform
        from vertexai.language_models import TextEmbeddingModel

        aiplatform.init(project=project_id, location=location)
        model = TextEmbeddingModel.from_pretrained("textembedding-gecko@003")

        with open(chunks.path) as f:
            all_chunks = json.load(f)

        results = []
        # 批次處理（每批最多 250 個）
        for i in range(0, len(all_chunks), 250):
            batch = all_chunks[i:i+250]
            texts = [c["text"] for c in batch]
            embs = model.get_embeddings(texts)

            for chunk, emb in zip(batch, embs):
                results.append({
                    "id": chunk["id"],
                    "embedding": emb.values,
                    "metadata": {"source": chunk["source"]}
                })

        with open(embeddings.path, "w") as f:
            json.dump(results, f)

        print(f"生成 {len(results)} 個 embeddings")

    # ── Pipeline 定義 ─────────────────────────────────────────
    @pipeline(
        name="rag-reindex-pipeline",
        description="RAG 知識庫自動重新索引管線"
    )
    def rag_reindex_pipeline(
        gcs_bucket: str,
        gcs_prefix: str = "documents/",
        project_id: str = GCP_PROJECT_ID,
        location: str = GCP_LOCATION
    ):
        load_task = load_documents(
            gcs_bucket=gcs_bucket,
            gcs_prefix=gcs_prefix
        )

        chunk_task = chunk_documents(
            documents=load_task.outputs["documents"]
        )

        embed_task = generate_embeddings(
            chunks=chunk_task.outputs["chunks"],
            project_id=project_id,
            location=location
        )

    print("Pipeline 定義完成")
    print("使用 kfp.compiler.Compiler().compile() 編譯後提交至 Vertex AI")

In [ ]:
# ============================================================
# [GCP] 使用 Cloud Scheduler 定期觸發 Pipeline
# 需要 GCP 專案與認證
# ============================================================

SCHEDULER_COMMANDS = f"""
# ── 步驟 1：編譯 Pipeline ─────────────────────────────────────
python -c "
from kfp.compiler import Compiler
from pipeline import rag_reindex_pipeline
Compiler().compile(rag_reindex_pipeline, 'pipeline.json')
"

# ── 步驟 2：建立 Pipeline 排程 ───────────────────────────────
gcloud ai pipelines schedules create \\
  --display-name="daily-rag-reindex" \\
  --schedule="0 2 * * *" \\
  --pipeline-definition-uri="gs://{GCP_PROJECT_ID}-pipelines/rag-reindex.json" \\
  --region="{GCP_LOCATION}"

# 或使用 Python SDK
from google.cloud import aiplatform
schedule = aiplatform.PipelineJobSchedule.create(
    display_name="daily-rag-reindex",
    cron="0 2 * * *",  # 每日凌晨 2:00 執行
    pipeline_definition_uri="gs://your-bucket/pipeline.json",
    parameter_values={{"gcs_bucket": "your-docs-bucket"}}
)
"""

print("Cloud Scheduler 設定：")
print(SCHEDULER_COMMANDS)

In [ ]:
# ============================================================
# [本地] 模擬 Pipeline 的 Python 腳本
# ============================================================

if not USE_GCP:
    import json
    import time

    class LocalReindexPipeline:
        """模擬 Vertex AI Pipeline 的本地版本"""

        def __init__(self):
            from openai import OpenAI
            import chromadb
            self.client = OpenAI(api_key=OPENAI_API_KEY)
            self.chroma = chromadb.Client()

        def load_documents(self, texts: list[dict]) -> list[dict]:
            """Component 1: 載入文件"""
            print(f"  [1/4] 載入 {len(texts)} 份文件")
            return texts

        def chunk_documents(self, docs: list[dict], chunk_size=200) -> list[dict]:
            """Component 2: 切塊"""
            chunks = []
            for doc in docs:
                text = doc["content"]
                for i, start in enumerate(range(0, len(text), chunk_size - 20)):
                    chunk = text[start:start + chunk_size]
                    if len(chunk) > 30:
                        chunks.append({"id": f"{doc['id']}_c{i}", "text": chunk})
            print(f"  [2/4] 產生 {len(chunks)} 個切塊")
            return chunks

        def generate_embeddings(self, chunks: list[dict]) -> list[dict]:
            """Component 3: 生成 Embeddings"""
            results = []
            for chunk in chunks:
                emb = self.client.embeddings.create(
                    model="text-embedding-3-small", input=chunk["text"]
                ).data[0].embedding
                results.append({"id": chunk["id"], "text": chunk["text"], "embedding": emb})
            print(f"  [3/4] 生成 {len(results)} 個 embeddings")
            return results

        def upsert_to_index(self, embeddings: list[dict], collection_name: str):
            """Component 4: 更新索引"""
            col = self.chroma.get_or_create_collection(collection_name)
            for item in embeddings:
                col.upsert(
                    ids=[item["id"]],
                    embeddings=[item["embedding"]],
                    documents=[item["text"]]
                )
            print(f"  [4/4] 已 upsert {len(embeddings)} 個向量到索引")

        def run(self, docs: list[dict]):
            """執行完整管線"""
            start = time.time()
            print("\n=== 執行重新索引管線 ===")
            loaded = self.load_documents(docs)
            chunks = self.chunk_documents(loaded)
            embeddings = self.generate_embeddings(chunks)
            self.upsert_to_index(embeddings, "rag_knowledge_base")
            elapsed = time.time() - start
            print(f"\n管線完成，耗時 {elapsed:.2f} 秒")

    # 執行管線
    pipeline = LocalReindexPipeline()
    sample_docs = [
        {"id": "doc_001", "content": "台積電 2024 年資本支出計畫達 280 億至 320 億美元，主要用於先進製程產能擴充。"},
        {"id": "doc_002", "content": "NVIDIA H100 GPU 採用台積電 4nm 製程製造，是目前 AI 訓練最主流的硬體平台。"}
    ]
    pipeline.run(sample_docs)

---
## Part 6：Google ADK 簡介

### 什麼是 Google Agent Development Kit (ADK)？

Google ADK（Agent Development Kit）是 Google 於 2024 年發布的**開源 Agent 框架**，專為建構企業級 AI Agent 系統設計。

### 核心概念

```
┌─────────────────────────────────────────────────────┐
│                  ADK 核心架構                        │
├─────────────────────────────────────────────────────┤
│                                                     │
│  ┌─────────┐  管理  ┌─────────┐  使用  ┌─────────┐ │
│  │ Runner  │──────▶│  Agent  │──────▶│  Tools  │ │
│  │         │       │         │       │         │ │
│  │ 執行循環 │       │ 決策邏輯 │       │ RAG搜索 │ │
│  │ 處理事件 │       │ Gemini  │       │ API呼叫 │ │
│  └─────────┘       └─────────┘       │ 計算工具 │ │
│       │                              └─────────┘ │
│       ▼                                          │
│  ┌─────────┐                                     │
│  │ Session │  儲存對話歷史與狀態                  │
│  └─────────┘                                     │
└─────────────────────────────────────────────────────┘
```

### ADK vs LangGraph 比較

| 面向 | Google ADK | LangGraph (NB09) |
|------|------------|------------------|
| 設計哲學 | Agent-first（以 Agent 為核心）| Graph-first（以工作流程為核心）|
| 主要模型 | Gemini（Vertex AI）| 模型無關 |
| 狀態管理 | Session 物件 | StateGraph 節點 |
| 多 Agent | 內建 Multi-Agent 支援 | 需自行設計節點 |
| 部署整合 | 深度整合 GCP（Cloud Run、Vertex AI）| 框架無關 |
| 監控 | 原生整合 Google Cloud Trace | LangSmith（需額外設定）|
| 學習曲線 | 較平（概念較直覺）| 較陡（Graph 概念需學習）|
| 開源程度 | 開源，但 GCP 整合深 | 完全開源 |

**何時選 ADK？**
- 整個系統已在 GCP 上
- 主要使用 Gemini 模型
- 需要快速建立多 Agent 協作系統

**何時選 LangGraph？**
- 需要精細控制工作流程（條件分支、循環）
- 模型多元（OpenAI + Anthropic + Gemini）
- 團隊熟悉 Graph 思維

In [ ]:
# ============================================================
# Google ADK 安裝與基本設定
# 注意：ADK 概念程式碼，不需要實際 GCP 認證即可閱讀理解
# ============================================================

# 安裝 ADK
# !pip install google-adk

print("Google ADK 版本資訊（截至 2024 年底）：")
print("  套件名稱：google-adk")
print("  主要依賴：google-cloud-aiplatform, vertexai")
print("  GitHub：https://github.com/google/adk-python")

In [ ]:
# ============================================================
# ADK 基本 Agent 範例 — RAG Agent
# 展示 ADK 的核心 API（概念程式碼）
# ============================================================

ADK_AGENT_CODE = '''
# adk_rag_agent.py
# Google ADK RAG Agent 範例

from google.adk.agents import Agent
from google.adk.tools import FunctionTool
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# ── 步驟 1：定義 Tools ────────────────────────────────────────

def search_knowledge_base(query: str, top_k: int = 3) -> dict:
    """
    搜尋 RAG 知識庫。

    Args:
        query: 搜尋查詢字串
        top_k: 回傳最相關的文件數量（預設 3）

    Returns:
        包含相關文件片段的字典
    """
    # 實際實作：呼叫 Vertex AI Vector Search 或 RAG Engine
    # 此處為示範用的假資料
    mock_results = [
        {"content": "台積電是全球最大晶圓代工廠", "score": 0.95},
        {"content": "台積電 3nm 製程效能領先三星", "score": 0.88},
    ]

    return {
        "query": query,
        "results": mock_results[:top_k],
        "total_found": len(mock_results)
    }

def get_latest_news(company: str) -> dict:
    """
    取得公司最新新聞。

    Args:
        company: 公司名稱（例如："台積電", "TSMC"）

    Returns:
        最新新聞摘要
    """
    # 實際實作：呼叫新聞 API
    return {
        "company": company,
        "news": ["台積電宣布擴大亞利桑那廠投資", "台積電 Q3 營收創新高"]
    }

# ── 步驟 2：建立 Agent ────────────────────────────────────────

rag_agent = Agent(
    name="semiconductor_research_agent",
    model="gemini-1.5-flash",   # 指定使用 Gemini 模型
    description="半導體產業研究助理，可搜尋知識庫並提供深度分析",
    instruction="""
        你是一位專業的半導體產業研究分析師。
        當使用者詢問問題時：
        1. 先使用 search_knowledge_base 工具搜尋相關資訊
        2. 如需最新動態，使用 get_latest_news 工具
        3. 綜合資訊提供準確、有深度的分析
        4. 如果不確定，誠實說明資訊的限制
        請用繁體中文回答。
    """,
    tools=[
        FunctionTool(search_knowledge_base),
        FunctionTool(get_latest_news),
    ]
)

# ── 步驟 3：建立 Runner 與 Session ────────────────────────────

session_service = InMemorySessionService()

runner = Runner(
    agent=rag_agent,
    app_name="semiconductor_research",
    session_service=session_service
)

# ── 步驟 4：執行查詢 ──────────────────────────────────────────

async def run_query(user_query: str, user_id: str = "user_001"):
    session = await session_service.create_session(
        app_name="semiconductor_research",
        user_id=user_id
    )

    message = types.Content(
        role="user",
        parts=[types.Part(text=user_query)]
    )

    # 執行 Agent（自動處理 Tool Calling 循環）
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=message
    ):
        if event.is_final_response():
            return event.content.parts[0].text

# 執行範例
# import asyncio
# result = asyncio.run(run_query("台積電在先進製程的競爭優勢是什麼？"))
# print(result)
'''

print("=== Google ADK RAG Agent 程式碼結構 ===")
print()
print("主要元件：")
print("  1. FunctionTool    - 將 Python 函數包裝為 Agent 可呼叫的工具")
print("  2. Agent           - 定義 Agent 行為（model、指令、工具）")
print("  3. Runner          - 管理 Agent 執行循環與事件處理")
print("  4. SessionService  - 管理對話歷史與使用者狀態")
print()
print("執行流程：")
print("  使用者問題 → Runner → Agent（Gemini）決定呼叫哪個 Tool")
print("           → Tool 執行 → 結果回傳 Agent → 生成最終回答")

print("\n完整程式碼如下：")
print(ADK_AGENT_CODE)

In [ ]:
# ============================================================
# ADK 多 Agent 協作範例（概念程式碼）
# ============================================================

ADK_MULTI_AGENT_CODE = '''
# 多 Agent 協作：研究 Agent 協調多個專業 Agent

from google.adk.agents import Agent

# 專業子 Agent
rag_retrieval_agent = Agent(
    name="rag_retrieval",
    model="gemini-1.5-flash",
    instruction="負責從知識庫中檢索相關資訊",
    tools=[FunctionTool(search_knowledge_base)]
)

news_agent = Agent(
    name="news_analyst",
    model="gemini-1.5-flash",
    instruction="負責搜尋並分析最新新聞",
    tools=[FunctionTool(get_latest_news)]
)

# 協調 Agent（Orchestrator）
orchestrator_agent = Agent(
    name="research_orchestrator",
    model="gemini-1.5-pro",   # 使用更強的模型做協調
    instruction="""
        你是研究協調者。根據使用者問題：
        1. 指派 rag_retrieval 搜尋知識庫背景資訊
        2. 指派 news_analyst 取得最新動態
        3. 整合兩者資訊，提供全面分析報告
    """,
    # ADK 內建的 Agent-as-Tool 功能
    tools=[
        rag_retrieval_agent.as_tool(),  # 將子 Agent 作為工具
        news_agent.as_tool(),
    ]
)
'''

print("=== ADK 多 Agent 協作架構 ===")
print()
print("""
使用者問題
     │
     ▼
research_orchestrator（協調 Agent）
     │
     ├──▶ rag_retrieval（子 Agent）→ 知識庫搜尋結果
     │
     └──▶ news_analyst（子 Agent）→ 最新新聞摘要
     │
     ▼
整合分析報告（最終回應）
""")
print("ADK 特點：Agent.as_tool() 讓多 Agent 協作變得非常簡潔")

---
## Part 7：面試 Q&A

以下整理了面試中常見的 GCP RAG 相關問題，涵蓋架構決策、服務選擇與最佳實踐。

### Q1：在設計 GCP 上的 RAG 系統時，你如何在 Vertex AI RAG Engine、自建 RAG + Vector Search、以及 AlloyDB pgvector 之間做選擇？

**A：依據以下決策框架選擇：**

**選 Vertex AI RAG Engine 的場景：**
- 需要快速 PoC 或 MVP（開發時間 < 2 週）
- 文件格式多樣（PDF、Docs、HTML），希望系統自動處理解析
- 團隊沒有 MLOps 資源維護向量索引管線
- 可接受黑箱行為，不需要深度客製化

**選自建 RAG + Vertex AI Vector Search 的場景：**
- 向量規模超過 10M，需要 HNSW 的高效能查詢
- 需要自訂切塊策略（例如：依據文件結構切塊而非字元數）
- 需要 Hybrid Search（向量搜索 + BM25 關鍵字搜索）
- 需要精細控制 Reranker 邏輯

**選 AlloyDB + pgvector 的場景：**
- 系統已有大量結構化資料在 PostgreSQL/AlloyDB
- 需要向量搜索與關聯式查詢組合（例如：「找出 2024 年後發布且與 AI 相關的文件」）
- 希望降低架構複雜度（不想維護獨立的向量資料庫）
- 向量規模在 1M 以內（pgvector 的效能上限）

**實際建議：** 大多數企業 RAG 系統初期用 RAG Engine 驗證，達到一定規模後遷移到自建方案以獲得更好的控制性。

### Q2：Vertex AI Vector Search 的 HNSW 索引與 IVF-Flat 有什麼區別？什麼情況下各自更優？

**A：兩者在精度-速度-記憶體之間有不同的取捨：**

**HNSW（Hierarchical Navigable Small World）：**
```
優點：
  • 查詢延遲低（< 5ms at p99）
  • 召回率高（通常 > 95%）
  • 支援動態插入（不需要重建整個索引）

缺點：
  • 記憶體使用量大（索引需常駐記憶體）
  • 建立索引時間較長
  • 費用較高（需要高記憶體機器）

適用場景：即時查詢、向量規模 < 100M、需要動態更新
```

**IVF-Flat（Inverted File Index）：**
```
優點：
  • 記憶體使用量小（支援磁碟索引）
  • 適合超大規模向量（> 1B）
  • 建立索引快

缺點：
  • 查詢延遲較高（需要掃描多個 cluster）
  • 召回率略低（取決於 nprobe 設定）
  • 不支援動態插入（需要週期性重建）

適用場景：批次查詢、超大規模、可接受略高延遲
```

**Vertex AI 的實際設定：** 使用 `create_tree_ah_index`（HNSW + Asymmetric Hashing）在多數生產場景中是最佳選擇，Google 內部做了大量優化。

### Q3：部署 RAG API 時，你如何在 Cloud Run vs GKE (Google Kubernetes Engine) 之間做選擇？

**A：主要依據流量特性、技術複雜度、成本進行決策：**

**選 Cloud Run 的場景：**
```
✅ 流量不穩定（有明顯的峰谷）→ Scale to Zero 節省費用
✅ 無狀態 API（每個請求獨立）
✅ 請求處理時間 < 60 分鐘
✅ 團隊沒有 Kubernetes 維運能力
✅ 快速部署需求（部署時間 < 5 分鐘）

限制：
❌ 冷啟動延遲（首次請求可能 2-5 秒）
❌ 無法做跨請求的記憶體共享（如 LRU 快取）
❌ 單一容器記憶體上限 32GB
```

**選 GKE 的場景：**
```
✅ 需要跨請求共享狀態（如 embedding 模型在記憶體中）
✅ 需要 GPU 支援（本地 Embedding 模型）
✅ 需要 Sidecar Pattern（日誌、監控代理）
✅ 複雜的微服務架構（多個 RAG 元件）
✅ 流量穩定且持續高（避免冷啟動）

限制：
❌ 維運複雜度高（需要 SRE 或 DevOps 專業）
❌ 最低費用高（至少需要 2-3 個節點）
❌ 部署週期較長
```

**實際建議：** RAG API 初期用 Cloud Run，當單一服務有持續 100+ QPS 且需要 GPU 本地 Embedding 時，才遷移至 GKE Autopilot。

### Q4：如何優化 GCP 上 RAG 系統的費用？你有哪些實際的成本控制策略？

**A：從以下幾個維度進行費用優化：**

**1. Embedding 費用優化**
```python
# ❌ 每次查詢都重新生成 embedding
emb = model.get_embeddings([query])

# ✅ 使用 Redis 快取常見查詢的 embedding
import hashlib
cache_key = f"emb:{hashlib.md5(query.encode()).hexdigest()}"
if cached := redis.get(cache_key):
    emb = json.loads(cached)
else:
    emb = model.get_embeddings([query])
    redis.setex(cache_key, 3600, json.dumps(emb))  # 快取 1 小時
```

**2. Vertex AI Vector Search 費用優化**
```
• 使用 Streaming Update（incremental）而非全量重建索引
• 設定適當的 min_replica_count（非生產環境設為 0）
• 使用 e2-standard 機器（比 n1-standard 便宜 30%）
• 定期清除過期向量（避免索引無限膨脹）
```

**3. LLM 費用優化**
```
• 對簡單查詢使用 Gemini Flash（比 Pro 便宜 10 倍）
• 啟用 Gemini 的 context caching（重複的系統 prompt）
• 設定合理的 max_output_tokens 上限
• 批次處理非即時請求（使用 Batch Prediction API）
```

**4. Cloud Run 費用優化**
```
• 啟用 min-instances=0（Scale to Zero）
• 使用 --cpu-throttling（CPU 只在處理請求時計費）
• 設定合理的並發數（避免過度配置）
```

**費用監控：** 設定 GCP Budget Alert，超過預算 80% 自動通知，避免月底驚喜帳單。

### Q5：ADK 與 LangGraph 的主要技術差異是什麼？在企業 GCP 環境中，你更傾向選哪個？

**A：兩者有根本性的設計哲學差異：**

**核心差異：**
```
LangGraph = 狀態機（State Machine）
  • 開發者明確定義「節點」和「邊」（狀態轉移）
  • 工作流程是靜態的（compile time 確定）
  • 適合有固定流程的任務（如：Review → Approve → Deploy）

ADK = 事件驅動（Event-Driven Agent Loop）
  • Agent 動態決定下一步（runtime 決策）
  • 工作流程是動態的（LLM 決定呼叫哪個 Tool）
  • 適合開放式任務（如："研究半導體市場並整理報告"）
```

**企業 GCP 環境的選擇建議：**

選 ADK 的理由：
- 原生整合 Vertex AI（IAM、VPC、Cloud Trace 開箱即用）
- `agent.as_tool()` 讓多 Agent 協作非常直覺
- Google 官方維護，與 Gemini 新功能同步更新
- 部署到 Cloud Run 的模板完善

選 LangGraph 的理由：
- 需要精確控制 Agent 的每一步（合規性稽核）
- 使用多家 LLM provider（不想鎖定 Google）
- 團隊已熟悉 LangChain 生態系
- 需要視覺化工作流程（LangGraph Studio）

**個人觀點：** 若系統完全在 GCP 且主要使用 Gemini，ADK 的整合優勢很明顯。若需要跨雲或多 LLM 策略，LangGraph 的模型無關性更有價值。

---

## 總結

本筆記本涵蓋了在 GCP 上建構企業級 RAG 系統的核心技術：

| Part | 核心學習 | 關鍵 API |
|------|----------|----------|
| 1 | Gemini vs OpenAI | `vertexai.init()`, `GenerativeModel()` |
| 2 | Vertex AI Vector Search | `MatchingEngineIndex`, `MatchingEngineIndexEndpoint` |
| 3 | Vertex AI RAG Engine | `rag.create_corpus()`, `rag.import_files()` |
| 4 | Cloud Run 部署 | `gcloud run deploy`, FastAPI |
| 5 | 自動化重新索引 | `kfp.dsl`, Vertex AI Pipelines |
| 6 | Google ADK | `Agent`, `Runner`, `FunctionTool` |

### 下一步學習資源

- [Vertex AI RAG Engine 官方文件](https://cloud.google.com/vertex-ai/generative-ai/docs/rag-overview)
- [Google ADK GitHub](https://github.com/google/adk-python)
- [Cloud Run 最佳實踐](https://cloud.google.com/run/docs/best-practices)
- [Vertex AI Vector Search 教學](https://cloud.google.com/vertex-ai/docs/vector-search/overview)